# E3 — Enrichers: ML control + Gemini LLM
Two enrichers, one shared interface (`predict -> pred_location/activity/companion`), scored
head-to-head in **E4**.

- **Control (non-LLM):** logistic regression (`SGDClassifier` log-loss, class-balanced) on the
  **full 225 features** — each method at its best (locked decision). Preprocessing (median impute +
  standardize) is fit **once per fold on train**; training rows capped at `TRAIN_CAP` for speed.
- **Experiment (LLM):** `gemini-3.1-flash-lite` via `google-genai`, reads a compact **text summary
  of interpretable descriptors**, returns a Pydantic `ContextLabel` (temperature=0, thinking=minimal),
  cached by prompt hash, per-call latency recorded.

Fairness: both scored on the **same stratified eval sample**; ML also reported on the full test set.

In [ ]:
import time, hashlib, json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

import ee_common as ee
ee.ensure_output_dirs()

gold = pd.read_parquet(ee.DATA_DIR / "e2_gold_labels.parquet")
folds = ee.load_folds()
TRAIN_CAP = 120_000              # cap train rows per head (linear model saturates well below this)
PER_CLASS_PER_FOLD = 40         # eval-sample size per gold-location class per fold -> drives LLM calls
print("gold rows:", len(gold), "| folds:", list(folds), "| LLM key:", ee.gemini_key_available())

## 1. Load features once, index by (uuid, timestamp)

In [ ]:
feat_df = ee.load_all()
FEATURES = ee.feature_cols(feat_df)
key = ["uuid", "timestamp"]
feat_df = feat_df.set_index(key).sort_index()
gold = gold.set_index(key).sort_index()
assert feat_df.index.equals(gold.index), "feature/gold alignment mismatch"
uid = feat_df.index.get_level_values("uuid")
print("aligned rows:", len(feat_df), "| features:", len(FEATURES))

## 2. ML control — preprocess once per fold, one logistic head per field

In [ ]:
def ml_fit_predict(fold):
    tr_u, te_u = folds[fold]["train"], folds[fold]["test"]
    Xtr_all = feat_df.loc[uid.isin(tr_u), FEATURES]
    Xte_all = feat_df.loc[uid.isin(te_u), FEATURES]
    imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(Xtr_all)
    sc = StandardScaler().fit(imp.transform(Xtr_all))
    Ztr = sc.transform(imp.transform(Xtr_all))          # preprocess ONCE (shared by 3 heads)
    Zte = sc.transform(imp.transform(Xte_all))
    out = pd.DataFrame(index=Xte_all.index)
    rng = np.random.default_rng(0)
    for f in ee.FIELDS:
        elig = gold.loc[Xtr_all.index, f"elig_{f}"].to_numpy()
        Zf, yf = Ztr[elig], gold.loc[Xtr_all.index[elig], f"gold_{f}"]
        if len(yf) > TRAIN_CAP:                          # random cap (balanced weights handle skew)
            sel = rng.choice(len(yf), TRAIN_CAP, replace=False)
            Zf, yf = Zf[sel], yf.iloc[sel]
        clf = SGDClassifier(loss="log_loss", class_weight="balanced",
                            max_iter=25, tol=1e-3, random_state=0).fit(Zf, yf)
        out[f"pred_{f}"] = clf.predict(Zte)
    out["fold"] = fold
    return out

t0 = time.time()
ml_pred = pd.concat([ml_fit_predict(k) for k in tqdm(folds, desc="ML folds")]).sort_index()
print(f"ML full-CV done in {time.time()-t0:.0f}s | predictions: {len(ml_pred):,}")

## 3. Common eval sample (stratified by gold location — keeps gym/beach)

In [ ]:
_LOC_ORD = {c: i for i, c in enumerate(
    ["home", "work", "gym", "beach", "restaurant", "transit", "outdoors", "unknown"])}

def eval_sample_for_fold(fold):
    g = gold.loc[uid.isin(folds[fold]["test"])]
    picked = []
    for cls, sub in g.groupby("gold_location"):
        take = min(PER_CLASS_PER_FOLD, len(sub))
        picked.append(sub.sample(take, random_state=1000 * int(fold) + _LOC_ORD.get(cls, 99)))
    return pd.concat(picked).index

parts = [eval_sample_for_fold(k) for k in folds]
eval_idx = parts[0].append(parts[1:]).unique()
eval_sample = gold.loc[eval_idx].copy()
fold_of = {u: k for k in folds for u in folds[k]["test"]}
eval_sample["fold"] = [fold_of[u] for u in eval_sample.index.get_level_values("uuid")]
print("eval sample:", len(eval_sample),
      "| by location:", eval_sample["gold_location"].value_counts().to_dict())
eval_sample.reset_index()[["uuid", "timestamp", "fold"]].to_parquet(
    ee.RESULTS_DIR / "e3_eval_sample.parquet", index=False)

## 4. LLM enricher — few-shot (from train users) → Gemini → `ContextLabel`
Enricher logic lives in `ee_enrichers.py` (importable, so LLM runs can be redone without retraining
ML). The LLM sees an interpretable text summary + **per-fold few-shot examples drawn only from that
fold's train users** (leakage-free; covers every location class incl. `unknown`, plus activities).

In [ ]:
import ee_enrichers as een

shots = een.fewshot_by_fold(gold, feat_df, folds, n_per_loc=2, n_per_act=1)
print("few-shot examples per fold:", {k: v.count(" => ") for k, v in shots.items()})
print(een.LLMEnricher(shots=shots)._prompt(feat_df.iloc[10], 0)[:600], "...")   # example prompt

## 5. Run LLM on the eval sample (only if a key is set)

In [ ]:
if ee.gemini_key_available():
    enr = een.LLMEnricher(shots=shots)
    llm_pred = enr.predict(feat_df.loc[eval_sample.index, :], eval_sample["fold"])
    llm_pred["fold"] = eval_sample["fold"].values
    llm_pred.reset_index().to_parquet(ee.RESULTS_DIR / "e3_pred_llm.parquet", index=False)
    n_err = llm_pred.get("_error", pd.Series(dtype=object)).notna().sum()
    lat = np.array(enr.latencies)
    cost = {"n_eval": int(len(llm_pred)), "n_live_calls": int(len(lat)),
            "mean_latency_s": float(lat.mean()) if len(lat) else 0.0,
            "parse_errors": int(n_err), "model": ee.GEMINI_MODEL, "mode": "few-shot"}
    json.dump(cost, open(ee.RESULTS_DIR / "e3_llm_cost.json", "w"), indent=2)
    print("LLM done:", cost)
else:
    print("No GEMINI_API_KEY -> skipped live LLM. Add a .env key and re-run this cell.")

## 6. Save ML predictions + quick sanity

In [ ]:
ml_pred.reset_index().to_parquet(ee.RESULTS_DIR / "e3_pred_ml.parquet", index=False)
from sklearn.metrics import f1_score
m = ml_pred.loc[eval_sample.index]
for f in ee.FIELDS:
    elig = eval_sample[f"elig_{f}"].to_numpy()
    y = eval_sample.loc[elig, f"gold_{f}"]
    print(f"ML macro-F1 [{f:9s}] on eval sample: "
          f"{f1_score(y, m.loc[y.index, f'pred_{f}'], average='macro', zero_division=0):.3f}")
print("saved:", ee.RESULTS_DIR / "e3_pred_ml.parquet")

**E3 done.** ML control trained + predicted across all folds; LLM enricher run with caching,
structured output, and latency capture. Predictions + cost saved for **E4** (full metric suite +
LLM-vs-ML comparison, bootstrap-CI verdict).